# Module 1: Pattern Matching & N-grams

## Learning Objectives
- Understand n-gram language models and Maximum Likelihood Estimation
- Implement a statistical language model on a tiny corpus
- Compute perplexity as an evaluation metric
- Connect n-gram concepts to Kyro's prefix caching


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import math
import random

print('Dependencies loaded.')

## Step 1: Build a Bigram Language Model

In [ ]:
corpus = (
    'It was the best of times it was the worst of times '
    'It was the age of wisdom it was the age of foolishness '
    'It was the epoch of belief it was the epoch of incredulity '
    'It was the season of light it was the season of darkness '
)

tokens = corpus.lower().split()
print(f'Corpus has {len(tokens)} tokens')
print(f'Unique tokens: {len(set(tokens))}')
print(f'Tokens: {tokens[:20]}')

In [ ]:
bigram_counts = defaultdict(lambda: defaultdict(int))
unigram_counts = defaultdict(int)

for i in range(len(tokens) - 1):
    prev = tokens[i]
    curr = tokens[i + 1]
    bigram_counts[prev][curr] += 1
    unigram_counts[prev] += 1

unigram_counts[tokens[-1]] += 1

print('Bigram transitions:')
for prev, nexts in bigram_counts.items():
    for curr, count in nexts.items():
        prob = count / unigram_counts[prev]
        print(f"  P('{curr}' | '{prev}') = {count}/{unigram_counts[prev]} = {prob:.4f}")

## Step 2: Compute Perplexity

In [ ]:
def compute_perplexity(bigram_counts, unigram_counts, tokens):
    log_prob_sum = 0.0
    n = 0
    for i in range(len(tokens) - 1):
        prev = tokens[i]
        curr = tokens[i + 1]
        prob = bigram_counts[prev][curr] / unigram_counts[prev]
        log_prob_sum += math.log2(prob + 1e-10)
        n += 1
    return 2 ** (-log_prob_sum / n)

ppl = compute_perplexity(bigram_counts, unigram_counts, tokens)
print(f'Perplexity (bigram): {ppl:.4f}')

## Step 3: Text Generation

In [ ]:
def generate_bigram(bigram_counts, unigram_counts, start_word, num_words=10):
    words = [start_word.lower()]
    current = start_word.lower()
    for _ in range(num_words):
        if current not in bigram_counts or not bigram_counts[current]:
            break
        nexts = dict(bigram_counts[current])
        total = sum(nexts.values())
        r = random.uniform(0, total)
        cumsum = 0
        for word, count in nexts.items():
            cumsum += count
            if cumsum >= r:
                words.append(word)
                current = word
                break
    return ' '.join(words)

print('Generated from "it":')
print(generate_bigram(bigram_counts, unigram_counts, 'it', 15))
print()
print('Generated from "the":')
print(generate_bigram(bigram_counts, unigram_counts, 'the', 15))

## Step 4: Add Smoothing (Laplace Add-1)

In [ ]:
vocab_size = len(set(tokens))

def compute_perplexity_smoothed(bigram_counts, unigram_counts, tokens, vocab_size):
    log_prob_sum = 0.0
    n = 0
    for i in range(len(tokens) - 1):
        prev = tokens[i]
        curr = tokens[i + 1]
        prob = (bigram_counts[prev][curr] + 1) / (unigram_counts[prev] + vocab_size)
        log_prob_sum += math.log2(prob + 1e-10)
        n += 1
    return 2 ** (-log_prob_sum / n)

ppl_smoothed = compute_perplexity_smoothed(bigram_counts, unigram_counts, tokens, vocab_size)
print(f'Perplexity (no smoothing): {ppl:.4f}')
print(f'Perplexity (Laplace smoothing): {ppl_smoothed:.4f}')
print('Smoothing reduces perplexity by handling unseen bigrams.')

## Exercises

1. **Implement trigram smoothing**: Extend the model to use trigrams (n=3) and compare perplexities.
2. **Compare corpus sizes**: Add more text to the corpus and observe how perplexity changes.
3. **Generate longer text**: Implement a back-off strategy when a bigram is not found.
4. **Kyro connection**: Explore `src/scheduler/radix_cache.rs` — how does the prefix matching logic resemble n-gram lookup?